# Notebook 2 — Keyword & Filtered Search

*Hands-on time: ~30 minutes*

Now that data from **all bids-examples datasets** is indexed, we explore ElasticSearch's
**structured query** capabilities:
1. Match queries (full-text / BM25)
2. Term queries (exact keyword match)
3. Range filters (numeric fields)
4. Bool compound queries (`must` / `should` / `filter`)
5. Aggregations (faceted analytics)

With dozens of datasets spanning different scanners, field strengths, and
institutions, we can see meaningful variation in search results.

**Prerequisites:** Complete Notebook 1.

In [1]:
from elasticsearch import Elasticsearch
import pandas as pd

client = Elasticsearch("http://localhost:9200", request_timeout=120)
INDEX_NAME = "neuroimaging"

count = client.count(index=INDEX_NAME)
print(f"Index '{INDEX_NAME}' has {count['count']} documents\n")

# Quick overview: how many docs per dataset?
overview = client.search(
    index=INDEX_NAME, size=0,
    aggs={"datasets": {"terms": {"field": "dataset"}}}
)
for b in overview["aggregations"]["datasets"]["buckets"]:
    print(f"  {b['key']:25s} {b['doc_count']:4d} docs")


def show_hits(response, fields=None):
    """Pretty-print search results as a DataFrame."""
    rows = []
    for hit in response["hits"]["hits"]:
        row = {"_score": hit["_score"]}
        src = hit["_source"]
        if fields:
            for f in fields:
                row[f] = src.get(f)
        else:
            # Exclude embedding for readability
            row.update({k: v for k, v in src.items() if k != "metadata_embedding"})
        rows.append(row)
    return pd.DataFrame(rows)

Index 'neuroimaging' has 4423 documents

  7t_trt                     439 docs
  ds000117                   395 docs
  ds108                      238 docs
  ds210                      225 docs
  ds006                      220 docs
  ds110                      216 docs
  ds009                      192 docs
  ds007                      158 docs
  ds113b                     156 docs
  ds107                      147 docs


---
## 1. Match Queries — Full-Text Search (BM25)

A `match` query analyzes the input text and uses BM25 scoring against
analyzed `text` fields like `description_text` and `SeriesDescription`.

In [2]:
# Full-text search over the description_text field
# With multiple datasets, results now come from different scanners and institutions
response = client.search(
    index=INDEX_NAME,
    query={
        "match": {
            "description_text": "anatomical structural T1"
        }
    },
    size=5
)

print(f"Hits: {response['hits']['total']['value']}")
show_hits(response, ["dataset", "suffix", "subject", "Manufacturer", "description_text"])

Hits: 859


,_score,dataset,suffix,subject,Manufacturer,description_text
0,7.491128,atlas-AAL,T1w,,None,T1-weighted anatomical structural MRI
1,7.491128,atlas-Destrieux,T1w,,None,T1-weighted anatomical structural MRI
2,7.491128,atlas-DiFuMo,T1w,,None,T1-weighted anatomical structural MRI
3,7.491128,atlas-HarvardOxford,T1w,,None,T1-weighted anatomical structural MRI
4,7.491128,atlas-Juelich,T1w,,None,T1-weighted anatomical structural MRI


In [3]:
# Multi-match: search across multiple text fields at once
response = client.search(
    index=INDEX_NAME,
    query={
        "multi_match": {
            "query": "Siemens functional brain",
            "fields": ["description_text", "SeriesDescription", "TaskName"],
            "type": "best_fields"
        }
    },
    size=5
)

print(f"Hits: {response['hits']['total']['value']}")
show_hits(response, ["dataset", "suffix", "subject", "Manufacturer", "TaskName"])

Hits: 2977


,_score,dataset,suffix,subject,Manufacturer,TaskName
0,6.351194,pet003,pet,01,Siemens,None
1,5.859378,pet001,pet,01,Siemens,None
2,5.859378,pet004,pet,01,Siemens,None
3,5.573025,pet002,pet,01,Siemens,None
4,5.573025,pet002,pet,01,Siemens,None


---
## 2. Term Queries — Exact Match on Keywords

For `keyword` fields, use `term` (single value) or `terms` (multiple values).
These are **exact** matches — no text analysis.

In [4]:
# Find all BOLD scans — note results span multiple datasets
response = client.search(
    index=INDEX_NAME,
    query={
        "term": {"suffix": "bold"}
    },
    size=5
)

print(f"BOLD hits: {response['hits']['total']['value']}")
show_hits(response, ["dataset", "subject", "task", "run", "RepetitionTime", "MagneticFieldStrength"])

BOLD hits: 2612


,_score,dataset,subject,task,run,RepetitionTime,MagneticFieldStrength
0,0.526737,7t_trt,01,rest,1,3.0,None
1,0.526737,7t_trt,01,rest,2,3.0,None
2,0.526737,7t_trt,01,rest,,4.0,None
3,0.526737,7t_trt,01,rest,1,3.0,None
4,0.526737,7t_trt,01,rest,2,3.0,None


In [5]:
# Filter by dataset — only scans from ds000117 (richest metadata)
response = client.search(
    index=INDEX_NAME,
    query={
        "term": {"dataset": "ds000117"}
    },
    size=5
)

print(f"ds000117 scans: {response['hits']['total']['value']}")
show_hits(response, ["dataset", "subject", "suffix", "Manufacturer", "MagneticFieldStrength", "InstitutionName"])

ds000117 scans: 395


,_score,dataset,subject,suffix,Manufacturer,MagneticFieldStrength,InstitutionName
0,2.414649,ds000117,01,T1w,Siemens,3.0,MRC Cognition and Brain Sciences Unit
1,2.414649,ds000117,01,FLASH,NaN,NaN,NaN
2,2.414649,ds000117,01,FLASH,NaN,NaN,NaN
3,2.414649,ds000117,01,FLASH,NaN,NaN,NaN
4,2.414649,ds000117,01,FLASH,NaN,NaN,NaN


In [6]:
# Match multiple values: T1w OR dwi — now spans multiple datasets with different scanners
response = client.search(
    index=INDEX_NAME,
    query={
        "terms": {"suffix": ["T1w", "dwi"]}
    },
    size=5
)

print(f"T1w or DWI hits: {response['hits']['total']['value']}")
show_hits(response, ["dataset", "subject", "suffix", "Manufacturer", "MagneticFieldStrength"])

T1w or DWI hits: 654


,_score,dataset,subject,suffix,Manufacturer,MagneticFieldStrength
0,1.0,2d_mb_pcasl,1,T1w,Siemens,3.0
1,1.0,7t_trt,01,T1w,NaN,NaN
2,1.0,7t_trt,02,T1w,NaN,NaN
3,1.0,7t_trt,03,T1w,NaN,NaN
4,1.0,7t_trt,04,T1w,NaN,NaN


---
## 3. Range Filters — Numeric Fields

Range queries let you filter by numeric values: TR, TE, field strength, etc.

In [7]:
# Find scans with RepetitionTime > 1.5 seconds
# With multiple datasets, we now have a wider range of TR values
response = client.search(
    index=INDEX_NAME,
    query={
        "range": {
            "RepetitionTime": {"gt": 1.5}
        }
    },
    size=5
)

print(f"TR > 1.5s: {response['hits']['total']['value']} hits")
show_hits(response, ["dataset", "subject", "suffix", "task", "RepetitionTime"])

TR > 1.5s: 2826 hits


,_score,dataset,subject,suffix,task,RepetitionTime
0,1.0,2d_mb_pcasl,1,T1w,,2.50
1,1.0,2d_mb_pcasl,1,epi,,8.00
2,1.0,2d_mb_pcasl,1,epi,,8.00
3,1.0,2d_mb_pcasl,1,asl,,3.58
4,1.0,7t_trt,01,bold,rest,3.00


In [8]:
# Filter by field strength — compare 1.5T vs 3T scans
for field_strength in [1.5, 3.0]:
    resp = client.search(
        index=INDEX_NAME,
        query={"term": {"MagneticFieldStrength": field_strength}},
        size=0
    )
    print(f"  {field_strength}T scans: {resp['hits']['total']['value']}")

# Range: scans at 3T or higher
response = client.search(
    index=INDEX_NAME,
    query={
        "range": {
            "MagneticFieldStrength": {"gte": 3.0}
        }
    },
    size=5
)

print(f"\n3T+ scans: {response['hits']['total']['value']} hits")
show_hits(response, ["dataset", "subject", "suffix", "MagneticFieldStrength", "Manufacturer"])

  1.5T scans: 15
  3.0T scans: 505

3T+ scans: 507 hits


,_score,dataset,subject,suffix,MagneticFieldStrength,Manufacturer
0,1.0,2d_mb_pcasl,1,T1w,3.0,Siemens
1,1.0,2d_mb_pcasl,1,epi,3.0,Siemens
2,1.0,2d_mb_pcasl,1,epi,3.0,Siemens
3,1.0,2d_mb_pcasl,1,asl,3.0,Siemens
4,1.0,asl001,Sub103,T1w,3.0,GE


---
## 4. Bool Compound Queries

The `bool` query combines clauses:
- **must**: All clauses must match (AND) — affects score
- **should**: At least one should match (OR) — boosts score
- **filter**: Must match — does NOT affect score (faster, cached)
- **must_not**: Must not match (exclusion)

In [9]:
# Complex query:
#   - MUST be a BOLD scan from a Siemens scanner
#   - FILTER by TR between 1.0 and 3.0 seconds
#   - SHOULD match "rest" in the task name (boosts score if present)
response = client.search(
    index=INDEX_NAME,
    query={
        "bool": {
            "must": [
                {"term": {"suffix": "bold"}},
                {"term": {"Manufacturer": "Siemens"}}
            ],
            "filter": [
                {"range": {"RepetitionTime": {"gte": 1.0, "lte": 3.0}}}
            ],
            "should": [
                {"match": {"TaskName": "rest"}}
            ]
        }
    },
    size=10
)

print(f"Compound query hits: {response['hits']['total']['value']}")
show_hits(response, ["dataset", "subject", "task", "run", "RepetitionTime", "Manufacturer", "MagneticFieldStrength"])

Compound query hits: 150


,_score,dataset,subject,task,run,RepetitionTime,Manufacturer,MagneticFieldStrength
0,4.521045,eeg_rest_fmri,32,rest,,2.160,Siemens,1.5
1,4.521045,eeg_rest_fmri,35,rest,,2.160,Siemens,1.5
2,4.521045,eeg_rest_fmri,36,rest,,2.160,Siemens,1.5
3,3.641261,atlas-4S,01,rest,,1.761,Siemens,3.0
4,3.641261,atlas-4S,01,rest,,1.761,Siemens,3.0
5,3.641261,atlas-4S,01,rest,,1.761,Siemens,3.0
6,0.910477,ds000117,01,facerecognition,01,2.000,Siemens,3.0
7,0.910477,ds000117,01,facerecognition,02,2.000,Siemens,3.0
8,0.910477,ds000117,01,facerecognition,03,2.000,Siemens,3.0
9,0.910477,ds000117,01,facerecognition,04,2.000,Siemens,3.0


In [10]:
# Cross-dataset: everything EXCEPT ds001 (show only metadata-rich datasets)
response = client.search(
    index=INDEX_NAME,
    query={
        "bool": {
            "must_not": [
                {"term": {"dataset": "ds001"}}
            ]
        }
    },
    size=5
)

print(f"Non-ds001 scans: {response['hits']['total']['value']}")
show_hits(response, ["dataset", "subject", "suffix", "Manufacturer", "MagneticFieldStrength", "InstitutionName"])

Non-ds001 scans: 4343


,_score,dataset,subject,suffix,Manufacturer,MagneticFieldStrength,InstitutionName
0,0.0,2d_mb_pcasl,1,T1w,Siemens,3.0,None
1,0.0,2d_mb_pcasl,1,epi,Siemens,3.0,None
2,0.0,2d_mb_pcasl,1,epi,Siemens,3.0,None
3,0.0,2d_mb_pcasl,1,asl,Siemens,3.0,None
4,0.0,7t_trt,01,T1map,NaN,NaN,None


---
## 5. Aggregations — Analytics & Faceted Counts

Aggregations let you compute statistics over your indexed data,
similar to SQL `GROUP BY`.

In [11]:
# Count documents by dataset (top-level view)
response = client.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "by_dataset": {
            "terms": {"field": "dataset"},
            "aggs": {
                "suffixes": {"terms": {"field": "suffix"}}
            }
        }
    }
)

print("Documents by dataset and scan type:")
for ds in response["aggregations"]["by_dataset"]["buckets"]:
    suffixes = ", ".join(f"{s['key']}({s['doc_count']})" for s in ds["suffixes"]["buckets"])
    print(f"  {ds['key']:25s} {ds['doc_count']:4d} docs — {suffixes}")

Documents by dataset and scan type:
  7t_trt                     439 docs — bold(132), magnitude1(88), phasediff(88), magnitude2(87), T1map(22), T1w(22)
  ds000117                   395 docs — FLASH(224), bold(144), T1w(16), dwi(11)
  ds108                      238 docs — bold(204), T1w(34)
  ds210                      225 docs — bold(225)
  ds006                      220 docs — bold(164), T1w(28), inplaneT2(28)
  ds110                      216 docs — bold(180), T1w(18), inplaneT2(18)
  ds009                      192 docs — bold(144), T1w(24), inplaneT2(24)
  ds007                      158 docs — bold(118), T1w(20), inplaneT2(20)
  ds113b                     156 docs — bold(156)
  ds107                      147 docs — bold(98), T1w(49)


In [12]:
# Stats on RepetitionTime grouped by MagneticFieldStrength
response = client.search(
    index=INDEX_NAME,
    size=0,
    query={"exists": {"field": "RepetitionTime"}},
    aggs={
        "by_field_strength": {
            "terms": {"field": "MagneticFieldStrength"},
            "aggs": {
                "tr_stats": {"stats": {"field": "RepetitionTime"}}
            }
        }
    }
)

print("RepetitionTime stats by field strength:")
for bucket in response["aggregations"]["by_field_strength"]["buckets"]:
    stats = bucket["tr_stats"]
    print(f"\n  {bucket['key']}T ({bucket['doc_count']} scans):")
    for k, v in stats.items():
        if v is not None:
            print(f"    {k}: {v}")

RepetitionTime stats by field strength:

  3.0T (408 scans):
    count: 408
    min: 0.006727200001478195
    max: 8.0
    avg: 2.306276872555506
    sum: 940.9609640026465

  1.5T (15 scans):
    count: 15
    min: 0.010999999940395355
    max: 8.300000190734863
    avg: 3.756400093436241
    sum: 56.34600140154362


In [13]:
# Nested aggregation: Manufacturer → scanner model → count
response = client.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "by_manufacturer": {
            "terms": {"field": "Manufacturer", "missing": "(unknown)"},
            "aggs": {
                "by_model": {
                    "terms": {"field": "ManufacturersModelName", "missing": "(unknown)"}
                },
                "unique_subjects": {
                    "cardinality": {"field": "subject"}
                }
            }
        }
    }
)

print("Scanner distribution:\n")
print(f"  {'Manufacturer':20s} {'Docs':>6s} {'Subjects':>10s}  Models")
print(f"  {'-'*70}")
for bucket in response["aggregations"]["by_manufacturer"]["buckets"]:
    models = ", ".join(m["key"] for m in bucket["by_model"]["buckets"])
    print(f"  {bucket['key']:20s} {bucket['doc_count']:6d} {bucket['unique_subjects']['value']:10.0f}  {models}")

Scanner distribution:

  Manufacturer           Docs   Subjects  Models
  ----------------------------------------------------------------------
  (unknown)              3889         66  (unknown)
  Siemens                 364         24  TrioTim, (unknown), Prisma, Skyra, Skyra Singo, Avanto , Prisma_fit, High-Resolution Research Tomograph (HRRT, CTI/Siemens), Biograph_mMR, Investigational_Device_7T
  Philips                 162         16  Achieva
  GE                        7          4  Discovery MR750, DISCOVERY MR750, DISCOVERY_MR750
  GE MEDICAL SYSTEMS        1          1  GE Advance


In [14]:
# Histogram: distribution of MagneticFieldStrength across all scans
response = client.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "field_strength_hist": {
            "histogram": {
                "field": "MagneticFieldStrength",
                "interval": 0.5
            }
        }
    }
)

print("MagneticFieldStrength distribution (0.5T bins):")
for bucket in response["aggregations"]["field_strength_hist"]["buckets"]:
    if bucket["doc_count"] > 0:
        bar = "█" * bucket["doc_count"]
        print(f"  {bucket['key']:5.1f}T: {bar} ({bucket['doc_count']})")

MagneticFieldStrength distribution (0.5T bins):
    1.5T: ███████████████ (15)
    3.0T: █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ (505)
    6.5T: ██ (2)


---
## Exercise

Try writing your own queries:

1. Find all scans from dataset `ds000117` that have a `Manufacturer` field
2. Find BOLD scans at 3T with TR < 2.5s — which datasets do they come from?
3. Compute the average FlipAngle per dataset (using a nested aggregation)

*(Solutions hidden below — scroll down after trying!)*

In [ ]:
# YOUR CODE HERE — Exercise 1

In [ ]:
# YOUR CODE HERE — Exercise 2

In [ ]:
# YOUR CODE HERE — Exercise 3

<details>
<summary><b>Click for solutions</b></summary>

```python
# Exercise 1
client.search(index=INDEX_NAME,
    query={
        "bool": {
            "must": [
                {"term": {"dataset": "ds000117"}},
                {"exists": {"field": "Manufacturer"}}
            ]
        }
    },
    size=10
)

# Exercise 2
client.search(index=INDEX_NAME,
    query={
        "bool": {
            "must": [
                {"term": {"suffix": "bold"}},
                {"term": {"MagneticFieldStrength": 3.0}}
            ],
            "filter": [{"range": {"RepetitionTime": {"lt": 2.5}}}]
        }
    },
    size=10
)

# Exercise 3
client.search(index=INDEX_NAME,
    size=0,
    query={"exists": {"field": "FlipAngle"}},
    aggs={
        "by_dataset": {
            "terms": {"field": "dataset"},
            "aggs": {
                "avg_flip_angle": {"avg": {"field": "FlipAngle"}}
            }
        }
    }
)
```
</details>

---
## Summary

You've learned:
- ✅ `match` and `multi_match` for full-text BM25 search
- ✅ `term` and `terms` for exact keyword matching
- ✅ `range` for numeric filtering
- ✅ `bool` queries to combine clauses with must/should/filter/must_not
- ✅ Aggregations: `terms`, `stats`, `cardinality`, `histogram`

**Next:** Open **Notebook 3** to unlock the power of **vector (semantic) search**.